# 🎨 Color Palette Extractor

Ever saved a Pinterest picture because the colors just looked *right*? 
Then spent 10 minutes manually color-picking each shade on your phone?

Same. So I automated it.

This notebook takes any image, finds its dominant colors using K-Means clustering,
gives each color a name, and scores how harmonious the palette is.

The same K-Means algorithm used for customer segmentation - just pointed at pixels instead of people. 🤷‍♀️

## Step 1: Get the libraries ready

Think of these as hiring a team of specialists:
- **PIL** = the photographer (opens and reads images)
- **numpy** = the mathematician (turns the image into numbers)
- **requests** = the delivery guy (fetches images from the internet)
- **BytesIO** = the translator (converts raw internet data into something PIL can actually read)
- **sklearn** = the ML brain (runs K-Means clustering)
- **matplotlib** = the artist (draws the pretty palette)
- **webcolors** = the dictionary (gives color names to hex codes)
- **scipy** = the calculator (measures distance between colors for harmony score)

In [ ]:
from PIL import Image
import numpy as np
import requests
from io import BytesIO
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import webcolors
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

print("All libraries loaded! Ready to go 🎨")

## Step 2: Load the image

Two options here:
- **Option A** (local file): Load from your own computer - works with any photo you've saved
- **Option B** (URL): Load from a direct image URL (must end in .jpg or .png - Instagram/Pinterest links won't work!)

Also converting RGBA → RGB here. Some PNGs have a 4th 'Alpha' channel (controls transparency).
We don't need it for color analysis, so we drop it.

In [ ]:
# OPTION A: Load from your computer (recommended!)
# Just save your Pinterest/Instagram image to your laptop and update the path below
img = Image.open(r"your_image_path_here.png")  # <-- change this!

# OPTION B: Load from a direct URL (uncomment these lines if using a URL)
# url = "https://your-direct-image-url.jpg"
# response = requests.get(url)
# img = Image.open(BytesIO(response.content))

# Convert to RGB (drops the Alpha channel if the image is RGBA)
# Think of Alpha as the opacity slider in Canva - we just don't need it here
img = img.convert('RGB')

# Turn the pretty image into a boring Excel sheet of numbers
# Each row = one pixel, each column = R, G, B value (0 to 255)
img_array = np.array(img)

print("Image shape:", img_array.shape)
print("Total pixels:", img_array.shape[0] * img_array.shape[1])
print("Each pixel has:", img_array.shape[2], "values (R, G, B)")

## Step 3: Flatten the image for K-Means

Right now the image is a 3D box: (height × width × 3)

K-Means needs a simple 2D table where every row is one pixel.

Imagine taking a multi-story apartment building and asking everyone to stand in one single line outside.
Same people, same groups of 3 (R, G, B), just arranged differently so K-Means can check them one by one.

The `-1` in reshape just means "figure out the number of rows yourself" - numpy is smart enough to calculate it!

In [ ]:
# Flatten from 3D (height x width x 3) to 2D (all pixels x 3)
pixels = img_array.reshape(-1, 3)

print("Before reshape:", img_array.shape)
print("After reshape:", pixels.shape)
print("Each row is one pixel - first 5 pixels:")
print(pixels[:5])

## Step 4: Remove background noise (optional but helpful!)

If the image has a plain grey or white background, those pixels will dominate the palette
and drown out the actual colors we care about.

Grey pixels = pixels where R, G, and B are all very similar to each other.
We filter those out before clustering so K-Means focuses on the real colors.

Same idea as data cleaning in ML - remove noise before training the model!

In [ ]:
# Convert pixels to a DataFrame so we can filter easily (Pandas to the rescue!)
pixels_df = pd.DataFrame(pixels, columns=['R', 'G', 'B'])

# Grey pixels = all three channels are very similar to each other
# If R ≈ G ≈ B, it's a grey/neutral - not what we're looking for
pixels_df['is_grey'] = (
    (abs(pixels_df['R'] - pixels_df['G']) < 20) &
    (abs(pixels_df['G'] - pixels_df['B']) < 20) &
    (abs(pixels_df['R'] - pixels_df['B']) < 20)
)

# Keep only the non-grey pixels for clustering
pixels_clean = pixels_df[~pixels_df['is_grey']][['R', 'G', 'B']].values

print(f"Original pixels: {len(pixels)}")
print(f"After removing grey background: {len(pixels_clean)}")
print(f"Removed {len(pixels) - len(pixels_clean)} background pixels")

## Step 5: Run K-Means to find dominant colors

This is the magic part.

K-Means looks at all the pixels and says:
*"Group these into N color families, find the most representative color of each family."*

It's the same algorithm used for customer segmentation at banks - just swapped customers for pixels!
K-Means doesn't care what the data points are. Give it pixels, it groups pixels.

- `n_clusters=8` → find 8 dominant colors (change this to get more or fewer colors)
- `random_state=42` → like a fixed shuffle - same results every time you run it
- `n_init=10` → tries 10 different starting positions, picks the best one (no bias from a bad starting point!)

In [ ]:
# Run K-Means on the clean pixels
# Each pixel is a data point with 3 features (R, G, B) - same structure as any ML problem!
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
kmeans.fit(pixels_clean)

# The cluster centers = the most representative color of each group
# .astype(int) because RGB values must be whole numbers (can't have R=211.7!)
dominant_colors = kmeans.cluster_centers_.astype(int)

print(f"{len(dominant_colors)} Dominant Colors found:")
for i, color in enumerate(dominant_colors):
    print(f"Color {i+1}: R={color[0]}, G={color[1]}, B={color[2]}")

## Step 6: Give the colors actual names

Hex codes are useful but 'mediumvioletred' is way more fun than '#df358b'.

The webcolors library maps RGB values to CSS color names.
If there's no exact match (which is almost always the case), we find the closest named color
using Euclidean distance - same distance formula from geometry class, just in 3D color space.

In [ ]:
def get_color_name(rgb):
    """Find the closest CSS3 color name for any RGB value."""
    try:
        # Try for an exact match first
        name = webcolors.rgb_to_name(tuple(rgb), spec=webcolors.CSS3)
        return name
    except ValueError:
        # No exact match - find the closest named color using distance
        min_distance = float('inf')
        closest_name = None
        for name in webcolors.names(spec=webcolors.CSS3):
            r, g, b = webcolors.name_to_rgb(name)
            # Euclidean distance in 3D color space - same as geometry class!
            distance = ((r - int(rgb[0]))**2 +
                       (g - int(rgb[1]))**2 +
                       (b - int(rgb[2]))**2) ** 0.5
            if distance < min_distance:
                min_distance = distance
                closest_name = name
        return closest_name

# Print all colors with their hex codes and names
print("Your color palette:")
for color in dominant_colors:
    hex_code = f'#{color[0]:02x}{color[1]:02x}{color[2]:02x}'
    name = get_color_name(color)
    print(f"  {hex_code} → {name}")

## Step 7: Draw the palette

Matplotlib is like MS Paint but in Python.
We're literally drawing colored rectangles on a canvas, one by one,
then labeling each with its hex code and name.

Note: Matplotlib needs colors between 0 and 1, not 0 and 255.
So we divide by 255 - like converting exam marks from a 255-point scale to a 0-1 scale.

In [ ]:
fig, ax = plt.subplots(1, figsize=(20, 3))
ax.set_xlim(0, len(dominant_colors))
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Dominant Color Palette', fontsize=14, pad=10)

for i, color in enumerate(dominant_colors):
    # Matplotlib needs 0-1 range, not 0-255
    rgb = tuple(c/255 for c in color)
    
    # Draw the color block
    rect = patches.Rectangle((i, 0), 1, 1, color=rgb)
    ax.add_patch(rect)
    
    # Add hex code label
    hex_code = f'#{color[0]:02x}{color[1]:02x}{color[2]:02x}'
    name = get_color_name(color)
    ax.text(i + 0.5, -0.12, hex_code, ha='center', fontsize=7)
    ax.text(i + 0.5, -0.22, name, ha='center', fontsize=7, style='italic')

plt.tight_layout()
plt.savefig('palette_named.png', dpi=150, bbox_inches='tight')
plt.show()
print("Palette saved as palette_named.png!")

## Step 8: Calculate the Harmony Score

How do you know if a palette is cohesive? Measure how far apart the colors are from each other.

Colors close together in RGB space = similar = cohesive = high score.
Colors far apart = very different = diverse = lower score.

Think of it like a dartboard 🎯 - hitting the centre gets full marks.
The further your dart lands from centre, the lower the score.

We use **Euclidean distance** in 3D color space.
Maximum possible distance = from pure black (0,0,0) to pure white (255,255,255) = ~441.

In [ ]:
# Calculate distance between every pair of colors
# cdist does this efficiently - measures all combinations at once
distances = cdist(dominant_colors, dominant_colors, 'euclidean')

# Average distance (ignoring zeros on the diagonal = distance from color to itself)
avg_distance = distances[distances > 0].mean()

# Convert to 0-100 score
# Lower distance = more cohesive = higher score (so we flip it with 1 - ...)
max_possible = 441  # max RGB distance: black (0,0,0) to white (255,255,255)
harmony_score = round((1 - avg_distance/max_possible) * 100, 1)

print(f"Average color distance: {avg_distance:.1f}")
print(f"Aesthetic Harmony Score: {harmony_score}/100")

if harmony_score >= 80:
    print("Verdict: Highly cohesive palette! 🌸")
elif harmony_score >= 60:
    print("Verdict: Moderately cohesive palette! 😊")
else:
    print("Verdict: Diverse/bold palette! 🎨")

## Full Summary Output

Everything together in one clean printout.

In [ ]:
print("=" * 45)
print("      COLOR PALETTE EXTRACTOR 🎨")
print("=" * 45)
print(f"\n🎨 Colors extracted: {len(dominant_colors)}")
print(f"\n🌈 Dominant Palette:")
for i, color in enumerate(dominant_colors):
    hex_color = f'#{color[0]:02x}{color[1]:02x}{color[2]:02x}'
    name = get_color_name(color)
    print(f"   Color {i+1}: RGB({color[0]}, {color[1]}, {color[2]}) → {hex_color} ({name})")
print(f"\n✨ Harmony Score: {harmony_score}/100")
if harmony_score >= 80:
    print("📊 Verdict: Highly cohesive palette! 🌸")
elif harmony_score >= 60:
    print("📊 Verdict: Moderately cohesive palette! 😊")
else:
    print("📊 Verdict: Diverse/bold palette! 🎨")
print("=" * 45)